<a href="https://colab.research.google.com/github/JonasLuizP/Magos-Tech/blob/main/servidor_modelo_ia(final).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install fastapi uvicorn pyngrok tensorflow pillow nest_asyncio

In [2]:
%%writefile main.py
from fastapi import FastAPI, Request
import tensorflow as tf
import numpy as np
from PIL import Image
import io

app = FastAPI()

model = tf.keras.models.load_model("/content/modelo_tumor_final.keras")

@app.post("/predict")
async def predict(request: Request):
    body = await request.body()

    try:
        img = Image.open(io.BytesIO(body)).convert("RGB")
    except:
        return {
            "status": "error",
            "label": "invalid_image",
            "confidence": 0,
            "probability": 0
        }

    # Pré-processamento correto (NÃO normalizar)
    img = img.resize((224,224))
    arr = np.array(img).astype("float32")
    arr = np.expand_dims(arr, axis=0)

    p = float(model.predict(arr, verbose=0)[0][0])

    if p >= 0.5:
        label = "🔴Tumor"
        confidence = p * 100
    else:
        label = "🟢Normal"
        confidence = (1 - p) * 100

    return {
        "status": "ok",
        "label": label,
        "confidence": round(confidence, 1),
        "probability": round(p, 4)
    }

Writing main.py


In [3]:
!nohup uvicorn main:app --host 0.0.0.0 --port 8000 &


nohup: appending output to 'nohup.out'


In [4]:
from pyngrok import ngrok
ngrok.set_auth_token("38tjYMAXqg7oGL4B93c5HLORyD6_4NKFoi2x9EucMJCC44P8V")


In [5]:
from pyngrok import ngrok
ngrok.kill()

url = ngrok.connect(8000)
print("URL pública:", url)


URL pública: NgrokTunnel: "https://vesicularly-untreatable-tyron.ngrok-free.dev" -> "http://localhost:8000"


In [ ]:
!pkill -f uvicorn
!pkill -f ngrok





In [6]:
!curl http://localhost:8000/docs


    <!DOCTYPE html>
    <html>
    <head>
    <link type="text/css" rel="stylesheet" href="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5/swagger-ui.css">
    <link rel="shortcut icon" href="https://fastapi.tiangolo.com/img/favicon.png">
    <title>FastAPI - Swagger UI</title>
    </head>
    <body>
    <div id="swagger-ui">
    </div>
    <script src="https://cdn.jsdelivr.net/npm/swagger-ui-dist@5/swagger-ui-bundle.js"></script>
    <!-- `SwaggerUIBundle` is now available on the page -->
    <script>
    const ui = SwaggerUIBundle({
        url: '/openapi.json',
    "dom_id": "#swagger-ui",
"layout": "BaseLayout",
"deepLinking": true,
"showExtensions": true,
"showCommonExtensions": true,
oauth2RedirectUrl: window.location.origin + '/docs/oauth2-redirect',
    presets: [
        SwaggerUIBundle.presets.apis,
        SwaggerUIBundle.SwaggerUIStandalonePreset
        ],
    })
    </script>
    </body>
    </html>
    